# Module 2 · Notebook 3 — PageRank: Ranking Nodes by Eigenvector Centrality
### Computing, Robotics & Bionics · adapted from Coursera *Mathematics for Machine Learning: Linear Algebra* (Imperial College London)

A **self-contained** case study that turns Notebook 2's eigenvectors into one of the most famous algorithms in computing: Google's original PageRank. The same machinery — a dominant eigenvector, found either directly or by repeatedly multiplying a matrix (**power iteration**) — is exactly what you'd use to find the most 'central' hub node in *any* network: a functional-connectivity graph of brain regions, a gene-regulatory network, a protein-interaction map. We finish by doing exactly that.

Run in **Google Colab** (*Runtime → Run all*). Everything is offline.

**Contents**
1. PageRank as an eigenvalue problem
2. Solving via eigenvectors
3. Power iteration
4. The damping parameter
5. A general `pageRank` function
6. Testing at scale
7. Bioengineering capstone: ranking hub regions in a brain network

Most sections end with an **Exercise**; run the **Solution** cell to check.

In [ ]:
import numpy as np
import numpy.linalg as la
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True, precision=4)
print("ready")

---
## 1 · PageRank as an eigenvalue problem

In the early 1990s, web search was dominated by manually curated directories. By the late 1990s Larry Page and Sergey Brin changed the game with a ranking algorithm rooted in linear algebra: imagine the web is browsed by random surfers who, from each page, follow one of its outgoing links at random. Pages that many other pages link to end up with more surfers visiting them — and a page is 'important' if *important* pages link to it, a recursive definition that turns out to be exactly an eigenvector equation.

We represent the number of surfers on each page with a vector **r**, and the transition rule with a matrix $L$ such that $\mathbf{r}^{(i+1)} = L\,\mathbf{r}^{(i)}$, where column $j$ of $L$ holds the probability of leaving page $j$ for each other page (so every column sums to 1 — $L$ is *column-stochastic*). Because of that structure, $1$ is always an eigenvalue of $L$ and every other eigenvalue has $|\lambda|\le 1$, so repeatedly applying $L$ converges to the steady state $L\,\mathbf{r} = \mathbf{r}$ — an eigenvector of $L$ with eigenvalue 1. That steady-state **r** *is* the PageRank.

**Our micro-internet.** Six pages, **A**–**F**, link to each other as follows (each page splits its outgoing links equally):
- **A** links to **B**, **C**, **D**
- **B** links to **A**, **C**
- **C** links to **F**
- **D** links to **A**, **C**, **E**
- **E** links to **F**
- **F** links to **C**, **D**

In [ ]:
# Utility: generate a random column-stochastic 'link matrix' for a network of n nodes
# (handy for testing at scale in Section 6 -- the exact recipe doesn't matter for now)
def generate_network(n):
    c = np.full([n, n], np.arange(n))
    c = (abs(np.random.standard_cauchy([n, n]) / 2) > (np.abs(c - c.T) + 1)) + 0
    c = (c + 1e-10) / np.sum((c + 1e-10), axis=0)
    return c

**Exercise 1.** Complete the link matrix `L` for the micro-internet described above: `L[i, j]` is the probability of moving from page `j` to page `i`, so **each column must sum to 1**. Once you've built `L`, keep it around: every section below uses it.

> 🤖 *Gemini tip:* "I need to build a column-stochastic transition matrix from a description of which pages link to which, splitting each page's outgoing probability equally among its links — walk me through it, and show me how to check that every column sums to 1."

In [ ]:
# Order: A, B, C, D, E, F

# Your code here

---
## 2 · Solving via eigenvectors

For a small system we can just ask NumPy for the eigenvalues/eigenvectors of `L` directly and read off the eigenvector for eigenvalue 1 (up to normalisation, it is the steady-state visitor distribution).

In [ ]:
eVals, eVecs = la.eig(L)
order = np.absolute(eVals).argsort()[::-1]     # order by eigenvalue magnitude
eVals, eVecs = eVals[order], eVecs[:, order]
r = eVecs[:, 0]                                  # principal eigenvector (eigenvalue closest to 1)
r = 100 * np.real(r / np.sum(r))                 # normalise to 100 'surfers' and drop tiny imag. part
pages = list("ABCDEF")
for p, v in sorted(zip(pages, r), key=lambda pv: -pv[1]):
    print(f"{p}: {v:6.2f}")

This method works, but for a network with millions of nodes computing a full eigendecomposition is far too slow, and we only ever need the *one* dominant eigenvector — a job **power iteration** is built for.

---
## 3 · Power iteration

Since repeatedly applying `L` drives every eigen-direction with $|\lambda|<1$ towards zero and leaves only the $\lambda=1$ direction, we can find the steady state just by multiplying: start from any distribution (say, 100 surfers split equally across the 6 pages) and keep applying `L` until it stops changing.

In [ ]:
n = L.shape[0]
r = 100 * np.ones(n) / n
for _ in range(100):
    r = L @ r
print("after 100 iterations:", r.round(2))

Or, better, iterate until the change is smaller than a tolerance rather than a fixed count:

In [ ]:
r = 100 * np.ones(n) / n
lastR = r
r = L @ r
i = 0
while la.norm(lastR - r) > 0.01:
    lastR = r
    r = L @ r
    i += 1
print(f"{i} iterations to convergence:", r.round(2))

---
## 4 · The damping parameter

The simple story above breaks down with a **trap**: suppose a new page **G** is added, linked to only by **F**, and **G** links only to itself. A surfer who reaches **G** can never leave — so run the power iteration on the extended network below and **G** ends up hoarding (almost) all the traffic, which is clearly not a sensible notion of 'importance'.

In [ ]:
# Extended network: F now also links to G (in addition to C, D); G is linked to by F,
# and only links to itself.
L2 = np.array([
    [0,   1/2, 0, 1/3, 0,   0,   0],
    [1/3, 0,   0, 0,   0,   0,   0],
    [1/3, 1/2, 0, 1/3, 0,   1/3, 0],
    [1/3, 0,   0, 0,   0,   1/3, 0],
    [0,   0,   0, 1/3, 0,   0,   0],
    [0,   0,   1, 0,   1,   0,   0],
    [0,   0,   0, 0,   0,   1/3, 1],
])
n2 = L2.shape[0]
r2 = 100 * np.ones(n2) / n2
for _ in range(100):
    r2 = L2 @ r2
print("A..F, G:", r2.round(2), "-- G has swallowed almost all the traffic!")

The fix: assume each surfer follows a link with probability $d$ (the **damping factor**) and, with probability $1-d$, jumps to a uniformly random page instead — escaping any trap. The transition matrix becomes

$$ M = d\,L + \frac{1-d}{n}\,J $$

where $J$ is the all-ones matrix. With $d=1$ we recover the pure link-following model; with $d=0$ every page is equally likely.

In [ ]:
d = 0.85
M = d * L2 + (1 - d) / n2 * np.ones([n2, n2])
r2 = 100 * np.ones(n2) / n2
lastR = r2
r2 = M @ r2
i = 0
while la.norm(lastR - r2) > 0.01:
    lastR = r2
    r2 = M @ r2
    i += 1
print(f"{i} iterations to convergence:", r2.round(2), "-- much more sensible now")

---
## 5 · A general `pageRank` function

Package the damped power-iteration recipe into a reusable function, so it works for a link matrix of *any* size.

**Exercise 2.** Complete `pageRank(linkMatrix, d)` below: it should build the damped matrix `M`, run power iteration from a uniform starting vector until convergence (tolerance `0.01`, as above), and return the normalised ranking vector.

> 🤖 *Gemini tip:* "Help me implement the power-iteration method to find the dominant eigenvector of a damped column-stochastic matrix, iterating until the change between steps drops below a tolerance, without calling np.linalg.eig."

In [ ]:
def pageRank(linkMatrix, d):
    """Return the (normalised) PageRank vector of linkMatrix, with damping factor d."""
# Your code here


def pageRankEigen(linkMatrix, d):
    """Same ranking, computed directly via the dominant eigenvector (for comparison/benchmarking)."""
    n = linkMatrix.shape[0]
    M = d * linkMatrix + (1 - d) / n * np.ones([n, n])
    eVals, eVecs = la.eig(M)
    order = np.absolute(eVals).argsort()[::-1]
    r = eVecs[:, order[0]]
    return r / la.norm(r)

# Once pageRank is complete, this should print True (loose atol: pageRank stops early,
# at a 0.01 change tolerance, while the eigenvector is exact)
print(
    "pageRank vs pageRankEigen agree on L2:",
    np.allclose(pageRank(L2, 0.85), np.abs(pageRankEigen(L2, 0.85)), atol=1e-2)
)

---
## 6 · Testing at scale

Power iteration's real payoff is speed on large networks, where a full eigendecomposition becomes impractical. Let's compare the two on a bigger, randomly generated network.

In [ ]:
L_big = generate_network(200)
%time r_power = pageRank(L_big, 0.85)
%time r_eigen = pageRankEigen(L_big, 0.85)
print("agree:", np.allclose(r_power, np.abs(r_eigen), atol=1e-2))

In [ ]:
plt.figure(figsize=(8, 3))
plt.bar(np.arange(len(r_power)), r_power)
plt.xlabel("node"); plt.ylabel("PageRank"); plt.title("PageRank across 200 randomly linked nodes")
plt.show()

---
## 7 · Bioengineering capstone: ranking hub regions in a brain network

Nothing about `pageRank` is specific to web pages: give it any column-stochastic matrix describing how 'influence' or 'signal' flows between nodes, and it ranks them by the same eigenvector-centrality idea. In network neuroscience this is exactly how researchers identify **hub regions** in a functional- or effective-connectivity graph — regions that many other regions feed into, and whose activity is disproportionately influential (or vulnerable, if damaged) across the network.

Consider a toy 6-region network — prefrontal cortex (**PFC**), primary motor cortex (**M1**), primary somatosensory cortex (**S1**), thalamus (**THAL**), hippocampus (**HIPP**), amygdala (**AMYG**) — with directed functional influence:

- **PFC** sends signal equally to **M1** and **HIPP**
- **M1** relays everything on to **S1**
- **S1** splits its output between **THAL** and back to **PFC**
- **THAL** (a classic relay hub) distributes equally to **PFC**, **M1**, and **S1**
- **HIPP** splits between **AMYG** and **PFC**
- **AMYG** feeds everything back into **HIPP**

**Exercise 3.** Build the column-stochastic influence matrix `Lbrain` for the network described above (region order `PFC, M1, S1, THAL, HIPP, AMYG`), then call `pageRank(Lbrain, d=0.85)` to rank the six regions by hub importance. Which region comes out on top, and does that match your intuition from the description (a region that many others feed into)?

> 🤖 *Gemini tip:* "I have a directed, weighted graph describing how signal flows between brain regions — help me turn it into a column-stochastic matrix and then rank the regions using the same eigenvector-centrality idea as PageRank."

In [ ]:
regions = ["PFC", "M1", "S1", "THAL", "HIPP", "AMYG"]

# Your code here

---
### PageRank complete
An eigenvalue problem (Notebook 2's eigenvectors, revisited), solved by power iteration instead of a full eigendecomposition, fixed with a damping term so dead ends and traps don't break it, and applied unchanged to a biological network. **Next:** Module 3 turns to calculus — derivatives, gradients, and integration.